<a href="https://colab.research.google.com/github/MichaelCM7/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1 - Identify the Task

To aid the self driving trailer make decisions based on it's environment.

**Goal**: Make safe maneuvering decisions on the road.

### Step 2 - Assemble Relevant Knowledge

The system needs to know the following things to be effective:
- React to traffic light colors.
- Detect and react to stop signs and speed limits.
- Detect Warning Signs.
- Detect road closures.
- Slow down when approaching Roundabouts, Potholes and speed humps.
- Detect if a vehicle is too close and slow down.
- Give way to emergency vehicles.
- Slow down and stop for pedestrians, children and animals.
- Slow down when there is heavy rain and fog.
- Spray sensor if the view is blocked.
- Stop at charge station or petrol station if the battery is low or the fuel is low.
- If it hits an obstacle it should make a noise.

### Step 3 - Define the Vocabulary

**Predicates**
  - TrafficLight(color)
  - RoadSign(type)
  - RoadCondition(hazard)
  - ObstacleDetected(object)
  - Weather(condition)
  - SensorStatus(state)
  - EnergyLevel(status),
  - CollisionDetected(event)
  - Action(state)
  - CanGo(state)
  - MustStop(state)
  - SlowDown(state)
  - Yield(state)
  - CleanSensor(state)
  - RefuelOrCharge(state)
  - SoundAlarm(event)

**Constants**
- **Traffic Lights:** Red, Yellow, Green
- **Signs and Closures:** StopSign, SpeedLimit, WarningSign, RoadClosure
- **Road Geometry and Surface:** Roundabout, Pothole, SpeedHump
- **Dynamic Road Users:** VehicleTooClose, EmergencyVehicle, Pedestrian, Child, Animal
- **Weather and Vehicle Status:** HeavyRain, Fog, SensorBlocked, LowBattery, LowFuel, ObstacleHit
- **Towns:** Nairobi, Thika, Murang'a, Nyeri, Nanyuki, Nakuru, Narok, Kericho, Bomet, Kisii, Kisumu

### Step 4 - Encode General Knowledge about the Domain

In [34]:
rules = {
    "Red": "Stop",
    "Yellow": "Slow down and prepare to stop",
    "Green": "Proceed",
    "StopSign": "Stop",
    "WarningSign": "Slow down and proceed with caution",
    "RoadClosure": "Stop and avoid the closed road",
    "Roundabout": "Slow down",
    "Pothole": "Slow down and avoid if safe",
    "SpeedHump": "Slow down",
    "VehicleTooClose": "Slow down and maintain safe distance",
    "EmergencyVehicle": "Give way",
    "Pedestrian": "Slow down or stop",
    "Child": "Slow down or stop",
    "Animal": "Slow down or stop",
    "HeavyRain": "Slow down",
    "Fog": "Slow down and proceed carefully",
    "SensorBlocked": "Activate sensor cleaning system",
    "LowBattery": "Stop at charging station",
    "LowFuel": "Stop at petrol station",
    "ObstacleHit": "Sound alarm"
}

cost_graph = {
    "Nairobi": [("Nakuru", 160), ("Narok", 141), ("Thika", 44)],
    "Thika": [("Murang'a", 46), ("Nyeri", 108)],
    "Murang'a": [("Nyeri", 54)],
    "Nyeri": [("Nakuru", 165), ("Nanyuki", 59)],
    "Nanyuki": [("Nakuru", 158)],
    "Nakuru": [("Nairobi", 160), ("Nyeri", 165), ("Narok", 123), ("Kericho", 107)],
    "Narok": [("Nairobi", 141)],
    "Kericho": [("Nakuru", 107), ("Bomet", 76), ("Kisumu", 83), ("Kisii", 90)],
    "Bomet": [("Narok", 82)],
    "Kisii": [("Bomet", 84)],
    "Kisumu": [("Kisii", 112)]
}

### Step 5 - Add New Facts to the Knowledge Base

In [35]:

# Display the current knowledge base
def show_kb():
    print("\nCurrent Knowledge Base")
    for situation, action in rules.items():
        print(f"{situation:20} -> {action}")

# Add a new fact directly to the knowledge base
def add_new_fact(situation, action):
    rules[situation] = action
    print(f"Added fact: '{situation}' -> '{action}'")

print("Add new facts (format: situation, action)")

# Allow the user to enter a new fact
def add_fact():
    new_situation = input("Enter a new road situation: ")
    new_action = input(f"What should the trailer do when {new_situation} is detected? ")

    rules[new_situation] = new_action
    print(f"New fact added: '{new_situation}' -> '{new_action}'")


# Add a fact using user input
add_fact()

# Display the updated knowledge base
show_kb()


Add new facts (format: situation, action)
Enter a new road situation: Train
What should the trailer do when Train is detected? Stop
New fact added: 'Train' -> 'Stop'

Current Knowledge Base
Red                  -> Stop
Yellow               -> Slow down and prepare to stop
Green                -> Proceed
StopSign             -> Stop
WarningSign          -> Slow down and proceed with caution
RoadClosure          -> Stop and avoid the closed road
Roundabout           -> Slow down
Pothole              -> Slow down and avoid if safe
SpeedHump            -> Slow down
VehicleTooClose      -> Slow down and maintain safe distance
EmergencyVehicle     -> Give way
Pedestrian           -> Slow down or stop
Child                -> Slow down or stop
Animal               -> Slow down or stop
HeavyRain            -> Slow down
Fog                  -> Slow down and proceed carefully
SensorBlocked        -> Activate sensor cleaning system
LowBattery           -> Stop at charging station
LowFuel          

### Step 6 - The current observation from sensors/camera

In [48]:
current_situation = input("Enter the current situation: ").capitalize()
starting_location = input("Enter starting town: ")
destination = input("Enter destination town: ")

Enter the current situation: pothole
Enter starting town: Nairobi
Enter destination town: Kisumu


### Step 7 - The agent queries the knowledge base

In [49]:
import heapq

def ucs(graph, start, goal):
    queue = [(0, [start])]
    visited = set()
    while queue:
        current = heapq.heappop(queue)
        current_cost, current_path = current[0], current[1]
        current_city = current_path[-1]
        if current_city == goal:
            return [current_path, current_cost]
        if current_city not in visited:
            visited.add(current_city)
            for location in graph.get(current_city, []):
                if location[0] not in visited:
                    heapq.heappush(queue, (current_cost + location[1], current_path + [location[0]]))
    return [None, -1]

def situation_advice(situation):
    for key, action in rules.items():
        if key.lower() == situation.lower():
            return action
    return "Unknown situation detected, proceed with caution or stop."

path, total_distance = ucs(cost_graph, starting_location, destination)
print(f"Best Path: {' - '.join(path)} ({total_distance} km)")
print(f"Observed situation: {current_situation}")
print("Agent Decision:", situation_advice(current_situation))

Best Path: Nairobi - Nakuru - Kericho - Kisumu (350 km)
Observed situation: Pothole
Agent Decision: Slow down and avoid if safe


### Step 8 - Querying the knowledge base

In [53]:
print("Agent Decision for 'Roundabout':", situation_advice("Roundabout"))
print("Agent Decision for 'EmergencyVehicle':", situation_advice("EmergencyVehicle"))
print("Agent Decision for 'SensorBlocked':", situation_advice("SensorBlocked"))

# Querying different locations
path1, dist1 = ucs(cost_graph, "Nairobi", "Kisumu")
print(f"Route Nairobi to Kisumu: {' - '.join(path1)} ({dist1} km)")
path2, dist2 = ucs(cost_graph, "Thika", "Nakuru")
print(f"Route Thika to Nakuru: {' - '.join(path2)} ({dist2} km)")
path3, dist3 = ucs(cost_graph, "Thika", "Narok")
print(f"Route Thika to Narok: {' - '.join(path3)} ({dist3} km)")

Agent Decision for 'Roundabout': Slow down
Agent Decision for 'EmergencyVehicle': Give way
Agent Decision for 'SensorBlocked': Activate sensor cleaning system
Route Nairobi to Kisumu: Nairobi - Nakuru - Kericho - Kisumu (350 km)
Route Thika to Nakuru: Thika - Murang'a - Nyeri - Nakuru (265 km)
Route Thika to Narok: Thika - Murang'a - Nyeri - Nakuru - Narok (388 km)


### Step 9 - Debugging the Knowledge Base

In [51]:
situations = ["LowFuel", "Red", "None", "ObstacleHit", "RoadKill", "RoadClosure", "GTA6", "Purple"]

for situation in situations:
    print(f"Situation: {situation:12} ->  Action: {situation_advice(situation)}")

Situation: LowFuel      ->  Action: Stop at petrol station
Situation: Red          ->  Action: Stop
Situation: None         ->  Action: Continue driving
Situation: ObstacleHit  ->  Action: Sound alarm
Situation: RoadKill     ->  Action: Unknown situation detected, proceed with caution or stop.
Situation: RoadClosure  ->  Action: Stop and avoid the closed road
Situation: GTA6         ->  Action: Unknown situation detected, proceed with caution or stop.
Situation: Purple       ->  Action: Unknown situation detected, proceed with caution or stop.


### Step 10: Add final reflections

Knowledge-based agents require a specific format of instructions following a case that was preselected, anything different may be interpreted as an entirely different instruction which it does not know.This exercise revealed the limitations of simple rules when it comes to guiding Knowledge-based agents, with insufficient parameters and incorrect formats being handled inadequately. Simulataneously occuring hazards are also difficult for the agent to determine the course of action.

### Team Photo

![Team Photo](team-photo.jpeg)